In [1]:
import glob
import json
import os
import re
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. TẢI MÔ HÌNH TEACHER TRỰC TIẾP TRÊN GPU KAGGLE
# ==========================================
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Đang tải Teacher {MODEL_ID} lên GPU...", flush=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)
model.eval()

# ==========================================
# 2. ĐỌC DỮ LIỆU TỪ READ-SPLITS-V1
# ==========================================
sub_files = glob.glob("/kaggle/input/**/teacher_sub_11k.parquet", recursive=True)
rl_files = glob.glob("/kaggle/input/**/rl_extra_pool.parquet", recursive=True)
assert len(sub_files) > 0, "Chưa gắn dataset read-splits-v1 vào Input!"

df_sub = pd.read_parquet(sub_files[0])
df_rl_extra = pd.read_parquet(rl_files[0])

# Lấy 5,500 mẫu cân bằng (250 Human / 250 AI mỗi miền)
sub_balanced = (
    df_sub.groupby(["domain", "label"], group_keys=False)
    .apply(lambda g: g.sample(n=min(250, len(g)), random_state=42))
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)
unused_sub = df_sub[~df_sub["sample_id"].isin(set(sub_balanced["sample_id"]))].reset_index(drop=True)

out_dir = "/kaggle/working/read_distilled"
os.makedirs(out_dir, exist_ok=True)
ckpt_path = f"{out_dir}/teacher_local_checkpoint.jsonl"

done_records = {}
if os.path.exists(ckpt_path):
    with open(ckpt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                done_records[rec["sample_id"]] = rec
            except Exception:
                continue

df_todo = sub_balanced[~sub_balanced["sample_id"].isin(done_records)].reset_index(drop=True)
print(f"Đã có sẵn: {len(done_records):,} | Cần chạy tiếp: {len(df_todo):,} mẫu", flush=True)

# ==========================================
# 3. SINH CHUỖI SUY LUẬN THEO BATCH (TẮT TQDM, IN LOG MỖI 200 MẪU)
# ==========================================
SYSTEM_PROMPT = (
    "You are an expert in textual forensics. Decide whether the given text is AI-generated or human-written. "
    "Provide a clear, evidence-based linguistic explanation inside <reasoning>...</reasoning> tags, "
    "followed by your final verdict inside <label>Human</label> or <label>AI</label> tags."
)

def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze the following text:\n\n{str(text)[:3500]}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_output(raw_text: str):
    r_match = re.search(r"<reasoning>(.*?)</reasoning>", raw_text, flags=re.DOTALL | re.IGNORECASE)
    l_match = re.search(r"<label>\s*(Human|AI)\s*</label>", raw_text, flags=re.IGNORECASE)
    reasoning = r_match.group(1).strip() if r_match else ""
    pred = l_match.group(1).capitalize() if l_match else None
    if pred == "Ai":
        pred = "AI"
    return reasoning, pred

BATCH_SIZE = 16
MAX_NEW_TOKENS = 384
LOG_EVERY = 200

if len(df_todo) > 0:
    total_todo = len(df_todo)
    processed = 0
    next_milestone = LOG_EVERY
    t0 = time.time()

    with open(ckpt_path, "a", encoding="utf-8") as f_out:
        for start in range(0, total_todo, BATCH_SIZE):
            batch_df = df_todo.iloc[start : start + BATCH_SIZE]
            prompts = [build_prompt(t) for t in batch_df["text"]]

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1536,
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_len = inputs["input_ids"].shape[1]
            for i, row in enumerate(batch_df.itertuples(index=False)):
                raw_resp = tokenizer.decode(outputs[i][input_len:], skip_special_tokens=True).strip()
                reasoning, pred_label = parse_output(raw_resp)

                rec = {
                    "sample_id": row.sample_id,
                    "text": row.text,
                    "label": row.label,
                    "domain": getattr(row, "domain", "unknown"),
                    "source": getattr(row, "source", "unknown"),
                    "raw_response": raw_resp,
                    "reasoning": reasoning,
                    "pred_label": pred_label,
                }
                f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                done_records[rec["sample_id"]] = rec

            f_out.flush()
            os.fsync(f_out.fileno())

            processed += len(batch_df)
            if processed >= next_milestone or processed == total_todo:
                elapsed_min = (time.time() - t0) / 60.0
                speed = processed / max(elapsed_min, 1e-6)
                eta_min = (total_todo - processed) / max(speed, 1e-6)
                print(
                    f"[Tiến độ] {processed:,}/{total_todo:,} ({processed/total_todo*100:.1f}%) | "
                    f"Đã chạy: {elapsed_min:.1f}p | Tốc độ: {speed:.1f} mẫu/p | Còn lại: ~{eta_min:.1f}p",
                    flush=True,
                )
                while next_milestone <= processed:
                    next_milestone += LOG_EVERY

# ==========================================
# 4. LỌC 2 LỚP -> XUẤT D_SFT.parquet & D_GRPO.parquet
# ==========================================
all_df = pd.DataFrame(list(done_records.values()))

def norm_label(val):
    return "Human" if str(val).strip().lower() in ["human", "1", "true"] else "AI"

all_df["gold_label"] = all_df["label"].apply(norm_label)

valid_fmt = (all_df["reasoning"].str.len() >= 40) & (all_df["pred_label"].isin(["Human", "AI"]))
valid_lbl = all_df["pred_label"] == all_df["gold_label"]

d_sft = all_df[valid_fmt & valid_lbl].copy().reset_index(drop=True)
d_sft["target_response"] = (
    "<reasoning>\n" + d_sft["reasoning"] + "\n</reasoning>\n<label>" + d_sft["gold_label"] + "</label>"
)

# Gộp phần sub bị lọc rớt + unused_sub + df_rl_extra thành tập D_GRPO
rejected_sub = all_df[~(valid_fmt & valid_lbl)][[c for c in df_sub.columns if c in all_df.columns]].copy()
d_grpo = pd.concat([rejected_sub, unused_sub, df_rl_extra], ignore_index=True)

sft_path = f"{out_dir}/D_SFT.parquet"
grpo_path = f"{out_dir}/D_GRPO.parquet"
d_sft.to_parquet(sft_path, index=False)
d_grpo.to_parquet(grpo_path, index=False)

print("-" * 60)
print(f"HOÀN TẤT! D_SFT: {len(d_sft):,} mẫu -> {sft_path}")
print(f"HOÀN TẤT! D_GRPO: {len(d_grpo):,} mẫu -> {grpo_path}")

Đang tải Teacher Qwen/Qwen2.5-7B-Instruct lên GPU...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Đã có sẵn: 0 | Cần chạy tiếp: 5,500 mẫu


/tmp/ipykernel_22/3273863120.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(250, len(g)), random_state=42))


[Tiến độ] 208/5,500 (3.8%) | Đã chạy: 9.4p | Tốc độ: 22.0 mẫu/p | Còn lại: ~240.0p
[Tiến độ] 400/5,500 (7.3%) | Đã chạy: 18.5p | Tốc độ: 21.6 mẫu/p | Còn lại: ~236.0p
[Tiến độ] 608/5,500 (11.1%) | Đã chạy: 28.2p | Tốc độ: 21.6 mẫu/p | Còn lại: ~226.8p
[Tiến độ] 800/5,500 (14.5%) | Đã chạy: 37.4p | Tốc độ: 21.4 mẫu/p | Còn lại: ~219.6p
[Tiến độ] 1,008/5,500 (18.3%) | Đã chạy: 46.7p | Tốc độ: 21.6 mẫu/p | Còn lại: ~208.1p
[Tiến độ] 1,200/5,500 (21.8%) | Đã chạy: 55.7p | Tốc độ: 21.5 mẫu/p | Còn lại: ~199.7p
[Tiến độ] 1,408/5,500 (25.6%) | Đã chạy: 65.4p | Tốc độ: 21.5 mẫu/p | Còn lại: ~190.2p
[Tiến độ] 1,600/5,500 (29.1%) | Đã chạy: 74.8p | Tốc độ: 21.4 mẫu/p | Còn lại: ~182.3p
[Tiến độ] 1,808/5,500 (32.9%) | Đã chạy: 84.7p | Tốc độ: 21.3 mẫu/p | Còn lại: ~172.9p
[Tiến độ] 2,000/5,500 (36.4%) | Đã chạy: 94.2p | Tốc độ: 21.2 mẫu/p | Còn lại: ~164.9p
[Tiến độ] 2,208/5,500 (40.1%) | Đã chạy: 103.5p | Tốc độ: 21.3 mẫu/p | Còn lại: ~154.4p
[Tiến độ] 2,400/5,500 (43.6%) | Đã chạy: 113.8p | Tốc